# 007 - SQL Runner with CodeMirror · 데모

**005 의 popup 자동완성 + 006 의 Python 콜백 실행** 을 한 번에 — CodeMirror 5.65.16 을 인라인 임베드해 "진짜 IDE" 같은 SQL 편집 경험.

## 005 / 006 / 007 한 눈에 비교

| 항목 | 005 (HTML/JS only) | 006 (ipywidgets) | **007 (CodeMirror 인라인)** |
|---|---|---|---|
| inline popup 자동완성 | ✅ | ❌ Button 칩 | ✅ Ctrl+Space + 자동 popup |
| 커서 위치 정밀 인서트 | ✅ | ❌ 끝에 append | ✅ 정확히 커서 위치 |
| 에디터 자체 syntax 색 | ❌ | ❌ 별도 미리보기 | ✅ 에디터 내부 컬러링 |
| line number / 라인 wrap | ❌ | ❌ | ✅ |
| ▶ 실행 → Python 콜백 | ❌ | ✅ | ✅ |
| 결과 자동 표 렌더 | ❌ | ✅ | ✅ |
| 단축키 실행 | ❌ | ❌ | ✅ Cmd/Ctrl+Enter |
| 의존성 | IPython | ipywidgets | ipywidgets + CM 인라인 (~250KB) |

## 시도할 것

- 에디터에 입력 시 `Ctrl+Space` (또는 식별자 입력 중 자동 popup) 으로 컨텍스트 인식 자동완성
- `FROM` 다음에는 테이블, `SELECT` / `WHERE` 다음에는 컬럼이 우선 추천
- `users.` 처럼 `테이블명.` 입력 → 해당 테이블 컬럼만 한정 추천
- 좌측 entity 트리의 컬럼 버튼 클릭 → CM 의 **현재 커서 위치** 에 정확히 인서트
- **Cmd/Ctrl + Enter** 로 ▶ 실행 → 아래 Output 영역에 DataFrame 표 출력

> ⚠ **Trusted notebook 필요**: JupyterLab 에서 처음 열면 "Untrusted" 표시될 수 있고, 이 경우 인라인 `<script>` 가 차단됩니다. 메뉴 → File → Trust Notebook 로 trust 해주세요.

---

## 1️⃣ 셋업: 임시 SQLite DB

In [ ]:
import os, sys, sqlite3, tempfile
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from sql_codemirror import SQLRunnerCM

db_path = os.path.join(tempfile.gettempdir(), 'sql_codemirror_demo.db')
if os.path.exists(db_path):
    os.remove(db_path)
with sqlite3.connect(db_path) as conn:
    conn.executescript('''
    CREATE TABLE users (
        id INTEGER PRIMARY KEY, name TEXT NOT NULL, region TEXT,
        signup_at TEXT
    );
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY, user_id INTEGER REFERENCES users(id),
        amount REAL, status TEXT, ordered_at TEXT
    );
    INSERT INTO users VALUES
        (1, '김알리스', '서울', '2024-01-15'),
        (2, '이밥',   '부산', '2024-02-20'),
        (3, '박찰리', '대구', '2024-03-10'),
        (4, '최다나', '서울', '2024-04-01');
    INSERT INTO orders VALUES
        (1, 1, 39000, 'paid',     '2024-05-01'),
        (2, 1, 12500, 'paid',     '2024-05-08'),
        (3, 2, 88000, 'paid',     '2024-05-12'),
        (4, 3, 15000, 'cancelled','2024-05-15'),
        (5, 1, 22000, 'paid',     '2024-06-02'),
        (6, 4, 99000, 'paid',     '2024-06-10'),
        (7, 2, 33000, 'pending',  '2024-06-15');
    ''')
print(f'✓ DB 준비 완료: {db_path}')

---

## 2️⃣ SQLRunnerCM 위젯 — 가장 짧은 형태

`with_sqlite()` 편의 메서드 한 줄로 thread-safe 하게 시작.

In [ ]:
runner = SQLRunnerCM.with_sqlite(db_path)
runner.set_query(
    "-- 사용자별 결제 합계 (status='paid' 만)\n"
    "SELECT u.name, u.region, SUM(o.amount) AS total\n"
    "FROM users u\n"
    "JOIN orders o ON o.user_id = u.id\n"
    "WHERE o.status = 'paid'\n"
    "GROUP BY u.name, u.region\n"
    "ORDER BY total DESC;"
)
runner.show()

**시도해보세요**:
1. 에디터에서 `Cmd/Ctrl + Enter` → 위 쿼리가 즉시 실행되어 DataFrame 표 렌더
2. 새 줄에 `SELECT name FROM ` 까지 타이핑 → `Ctrl+Space` → `users` / `orders` 추천
3. `users.` 까지만 입력 → `users.id`, `users.name`, `users.region`, `users.signup_at` 만 추천
4. 좌측 트리의 컬럼 버튼 클릭 → 에디터 **커서 위치에** 그대로 들어옴 (006 처럼 끝에 append 되지 않음)

---

## 3️⃣ pandas DataFrame 으로 직접 등록 — 콜백 echo

사내 분석 노트북에서 `on_execute` 를 `pandasql` / `duckdb` / 사내 엔진 등으로 연결할 수 있습니다. 데모에선 SQL 만 echo.

In [ ]:
df_events = pd.DataFrame({
    'event_id': range(1, 11),
    'user_id':  [1, 1, 2, 3, 1, 4, 2, 4, 3, 1],
    'event_type': ['click','view','click','purchase','view',
                   'click','view','purchase','view','click'],
    'value':    [None, None, None, 39000, None,
                 None, None, 55000, None, None],
})

def echo_only(sql):
    print('=== 받은 SQL ===')
    print(sql)
    print('=== 실행 엔진과 연결되면 결과가 여기에 출력됩니다 ===')

echo_runner = SQLRunnerCM(on_execute=echo_only)
echo_runner.from_dataframes({'events': df_events})
echo_runner.set_query('SELECT event_type, COUNT(*) FROM events GROUP BY event_type;')
echo_runner.show()

---

## 4️⃣ 콜백 없이 — 디자인 확인용

`on_execute` 미주입 시 ▶ 실행 클릭 → SQL 텍스트만 출력.

In [ ]:
preview = SQLRunnerCM()
preview.from_dict({
    'customers': ['id', 'name', 'phone'],
    'invoices':  [('id', 'INT'), ('customer_id', 'INT'), ('total', 'REAL')],
})
preview.set_query('SELECT * FROM customers;')
preview.show()

---

## 정리

- **에디터 내부 syntax highlight + popup 자동완성** — 005/006 가 따로 가졌던 것을 한 위젯에서 모두 제공
- **커서 위치 정밀 인서트** — CodeMirror 가 cursor API 를 노출하므로 가능 (Textarea 한계 우회)
- **Cmd/Ctrl + Enter 단축 실행** — 마우스 없이도 빠르게 반복 실행
- **인라인 임베드** — `_assets/` 폴더 없이 `sql_codemirror.py` 한 파일만 폐쇄망에 반입하면 됨
- **trade-off**: 파일 크기 ~270KB · CodeMirror MIT 라이선스 검토 필요

## 폐쇄망 친화 체크

| 항목 | 상태 |
|---|---|
| 외부 네트워크 / CDN | ❌ 없음 (모든 CSS/JS 인라인) |
| `<link href>` / `<script src>` | ❌ 없음 |
| 새 서버 / 포트 | ❌ 없음 |
| 바이너리 영속화 | ❌ 없음 |
| 단일 반입 단위 | `sql_codemirror.py` 한 파일 |
| 추가 패키지 | ipywidgets + IPython (이미 스택 포함) |